In [1]:
import time
from typing import Iterator

import numpy
from smodels.decomposition.theorySMS import TheorySMS
from smodels.base.genericSMS import GenericSMS
from smodels.decomposition.topologyDict import TopologyDict
from smodels.base.particleNode import ParticleNode
from smodels.base.physicsUnits import fb, GeV
from smodels.decomposition.exceptions import SModelSDecompositionError as SModelSError
from smodels.base.smodelsLogging import logger
from itertools import product
from smodels.base import runtime
from smodels.decomposition import decomposer
from smodels.base.physicsUnits import fb, GeV, TeV
from smodels.matching.theoryPrediction import theoryPredictionsFor,TheoryPredictionsCombiner
from smodels.experiment.databaseObj import Database
from smodels.base.smodelsLogging import setLogLevel
from smodels.tools.particlesLoader import load
from smodels.share.models.SMparticles import SMList
from smodels.base.particle import Particle, InvisibleParticle
from smodels.base.model import Model
from smodels.decomposition.decomposer import decompose
from smodels.decomposition.decomposerNew import decomposeNew
from smodels.decomposition.decomposerOld import decomposeOld
import itertools
import time
import numpy as np
setLogLevel("info")

In [2]:
# Load the BSM model
runtime.modelFile = "nmssmPoints/000.slha"
BSMList = load()
model = Model(BSMparticles=BSMList, SMparticles=SMList)
slhafile = 'nmssmPoints/022.slha'
model.updateParticles(inputFile=slhafile,ignorePromptQNumbers = ['eCharge','spin'])


INFO in model.updateParticles() in 428: Loaded 58 BSM particles


In [3]:
massCompress = True
invisibleCompress = True

# Set main options for decomposition
sigmacut = 0.005*fb
mingap = 10.*GeV
mingapISR = 10.0*GeV


In [4]:
t0 = time.time()
# topDict = decomposeOld(model, sigmacut,
#                         massCompress=massCompress, invisibleCompress=invisibleCompress,
#                         minmassgap=mingap, minmassgapISR=mingapISR)

topDict = decomposeNew(model, sigmacut,
                        massCompress=massCompress, invisibleCompress=invisibleCompress,
                        minmassgap=mingap, minmassgapISR=mingapISR)   

# topDict = decompose(model, sigmacut,
#                         massCompress=massCompress, invisibleCompress=invisibleCompress,
#                         minmassgap=mingap, minmassgapISR=mingapISR)       
print(f'Done in {time.time()-t0:.2f} seconds')
print(len(topDict),len(topDict.getSMSList()))    

#Done in 15.13 seconds
#38 16622

#Done in 34.96 seconds
#42 28087

Done in 31.76 seconds
42 27809


In [5]:
import cProfile
import io
import pstats

def _fresh_model():
    runtime.modelFile = "nmssmPoints/000.slha"
    bsm_list = load()
    fresh_model = Model(BSMparticles=bsm_list, SMparticles=SMList)
    fresh_model.updateParticles(inputFile=slhafile, ignorePromptQNumbers=['eCharge', 'spin'])
    return fresh_model

def _cumtime_for(stats_obj, needles):
    total = 0.0
    for (filename, _lineno, funcname), stat in stats_obj.stats.items():
        haystack = f"{filename}:{funcname}"
        if any(needle in haystack for needle in needles):
            total += stat[3]
    return total

profiler = cProfile.Profile()
profile_model = _fresh_model()
t0 = time.perf_counter()
profiler.enable()
profile_topDict = decomposeNew(
    profile_model,
    sigmacut,
    massCompress=massCompress,
    invisibleCompress=invisibleCompress,
    minmassgap=mingap,
    minmassgapISR=mingapISR,
 )
profiler.disable()
dt = time.perf_counter() - t0

stats = pstats.Stats(profiler)
stats_stream = io.StringIO()
stats.sort_stats("cumulative").stream = stats_stream
stats.print_stats(25)

print(f"profiled decomposeNew in {dt:.3f} s")
print(f"topologies={len(profile_topDict)} sms={len(profile_topDict.getSMSList())}")
print(stats_stream.getvalue())

hotspots = [
    ("decomposeNew total", ["decomposerNew.py:decomposeNew"]),
    ("build_subtree_cache", ["decomposerNew.py:build_subtree_cache"]),
    ("getDecayNodes", ["decomposerNew.py:getDecayNodes"]),
    ("cartesian product", ["itertools:product"]),
    ("TopologyDict.addSMS", ["topologyDict.py:addSMS"]),
    ("TheorySMS.copy", ["theorySMS.py:copy"]),
    ("setGlobalProperties", ["theorySMS.py:setGlobalProperties"]),
    ("compress total", ["topologyDict.py:compress", "theorySMS.py:compress"]),
    ("massCompress", ["theorySMS.py:massCompress"]),
    ("invisibleCompress", ["theorySMS.py:invisibleCompress"]),
]

print("Hotspot                           cumulative s")
print("-" * 54)
for label, needles in hotspots:
    print(f"{label:<32} {_cumtime_for(stats, needles):12.3f}")

INFO in model.updateParticles() in 428: Loaded 58 BSM particles


KeyboardInterrupt: 

In [ ]:
profiler = cProfile.Profile()
profile_model = _fresh_model()
t0 = time.perf_counter()
profiler.enable()
profile_topDict_nocomp = decomposeNew(
    profile_model,
    sigmacut,
    massCompress=False,
    invisibleCompress=False,
    minmassgap=mingap,
    minmassgapISR=mingapISR,
 )
profiler.disable()
dt_nocomp = time.perf_counter() - t0

stats_nocomp = pstats.Stats(profiler)
stats_stream = io.StringIO()
stats_nocomp.sort_stats("cumulative").stream = stats_stream
stats_nocomp.print_stats(20)

hotspots_nocomp = [
    ("decomposeNew total", ["decomposerNew.py:decomposeNew"]),
    ("build_subtree_cache", ["decomposerNew.py:build_subtree_cache"]),
    ("getDecayNodes", ["decomposerNew.py:getDecayNodes"]),
    ("TopologyDict.addSMS", ["topologyDict.py:addSMS"]),
    ("TheorySMS.copy", ["theorySMS.py:copy"]),
    ("setGlobalProperties", ["theorySMS.py:setGlobalProperties"]),
    ("compress total", ["topologyDict.py:compress", "theorySMS.py:compress"]),
]

print(f"profiled decomposeNew without compression in {dt_nocomp:.3f} s")
print(f"topologies={len(profile_topDict_nocomp)} sms={len(profile_topDict_nocomp.getSMSList())}")
print(stats_stream.getvalue())

print("Hotspot                           cumulative s")
print("-" * 54)
for label, needles in hotspots_nocomp:
    print(f"{label:<32} {_cumtime_for(stats_nocomp, needles):12.3f}")

INFO in model.updateParticles() in 428: Loaded 58 BSM particles


profiled decomposeNew without compression in 35.318 s
topologies=40 sms=15535
         48313638 function calls (46560001 primitive calls) in 35.023 seconds

   Ordered by: cumulative time
   List reduced from 220 to 20 due to restriction <20>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
      3/2    0.000    0.000   35.318   17.659 /home/lessa/.local/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3541(run_code)
        2    0.000    0.000   35.318   17.659 {built-in method builtins.exec}
        1    3.617    3.617   28.239   28.239 /home/lessa/smodels-main/smodels/decomposition/decomposerNew.py:25(decomposeNew)
    25541    0.472    0.000   15.558    0.001 /home/lessa/smodels-main/smodels/decomposition/topologyDict.py:25(addSMS)
    25541    0.112    0.000   11.665    0.000 /home/lessa/smodels-main/smodels/decomposition/theorySMS.py:98(setGlobalProperties)
   199466    0.495    0.000    9.161    0.000 /home/lessa/smodels-main/smodels/decompos

In [ ]:
import smodels.decomposition.decomposerNew as decomposerNew_mod

def timed_decomposeNew(model, sigmacut, massCompress, invisibleCompress, minmassgap, minmassgapISR):
    phase = {
        'xsec_sort': 0.0,
        'build_primary_sms': 0.0,
        'build_subtree_cache': 0.0,
        'top_level_product': 0.0,
        'copy_and_attach': 0.0,
        'set_global_properties': 0.0,
        'addSMS': 0.0,
        'compress': 0.0,
        'total': 0.0,
    }
    counters = {
        'production_sms': 0,
        'cache_particles': 0,
        'primary_combos_considered': 0,
        'primary_combos_kept': 0,
    }

    t_total = time.perf_counter()
    xSectionList = model.xsections
    sigmacutFB = sigmacut.asNumber(fb)

    t0 = time.perf_counter()
    xSectionList.removeLowerOrder()
    xSectionList.sort()
    phase['xsec_sort'] += time.perf_counter() - t0

    productionSMS = []
    smsTopDict = TopologyDict()

    t0 = time.perf_counter()
    for pdgs in xSectionList.getPIDpairs():
        weight = xSectionList.getXsecsFor(pdgs)
        maxWeight = weight.getMaxXsec().asNumber(fb)
        if maxWeight < sigmacutFB:
            continue
        pv = ParticleNode(model.getParticle(label='PV'))
        primaryMothers = [ParticleNode(model.getParticle(pdg=pdg)) for pdg in pdgs]
        newSMS = TheorySMS()
        newSMS.maxWeight = maxWeight
        newSMS.prodXSec = weight
        pvIndex = newSMS.add_node(pv)
        motherIndices = newSMS.add_nodes_from(primaryMothers)
        newSMS.add_edges_from(product([pvIndex], motherIndices))
        productionSMS.append(newSMS)
    phase['build_primary_sms'] += time.perf_counter() - t0
    counters['production_sms'] = len(productionSMS)

    maxXsec = max(sms.maxWeight for sms in productionSMS)
    minBR = sigmacutFB / maxXsec
    cache = {}

    t0 = time.perf_counter()
    for sms in productionSMS:
        for particleNode in sms.daughters(sms.rootIndex):
            if particleNode.particle is None:
                continue
            if particleNode.particle in cache:
                continue
            counters['cache_particles'] += 1
            _, cache = decomposerNew_mod.build_subtree_cache(particleNode, memo=cache, minBR=minBR)
    phase['build_subtree_cache'] += time.perf_counter() - t0

    daughter_combo_time = 0.0
    attach_time = 0.0
    global_time = 0.0
    addsms_time = 0.0

    for sms in productionSMS:
        all_subtrees = [
            cache.get(sms.indexToNode(daughterIndex).particle, [])
            for daughterIndex in sms.daughterIndices(sms.rootIndex)
        ]
        daughterIndices = list(sms.daughterIndices(sms.rootIndex))

        t_prod = time.perf_counter()
        for primary_subtrees in itertools.product(*all_subtrees):
            counters['primary_combos_considered'] += 1
            totalBR = 1.0
            for subtree in primary_subtrees:
                totalBR *= subtree.decayBRs
            if sms.maxWeight * totalBR < sigmacutFB:
                continue
            counters['primary_combos_kept'] += 1

            t_attach = time.perf_counter()
            smsDecayed = sms.copy()
            for idaughter, subtree in enumerate(primary_subtrees):
                old2newIndexMapping = {0: daughterIndices[idaughter]}
                for nodeIndex in subtree.nodeIndices:
                    if nodeIndex == subtree.rootIndex:
                        continue
                    node = subtree.indexToNode(nodeIndex)
                    newIndex = smsDecayed.add_node(node)
                    old2newIndexMapping[nodeIndex] = newIndex

                for edgeA, edgeB in subtree.edgeIndices:
                    smsDecayed.add_edge(old2newIndexMapping[edgeA], old2newIndexMapping[edgeB])
            smsDecayed.decayBRs = totalBR
            smsDecayed.maxWeight = sms.maxWeight * totalBR
            attach_time += time.perf_counter() - t_attach

            t_global = time.perf_counter()
            smsDecayed.setGlobalProperties()
            smsDecayed.ancestors = [smsDecayed]
            global_time += time.perf_counter() - t_global

            t_addsms = time.perf_counter()
            smsTopDict.addSMS(smsDecayed)
            addsms_time += time.perf_counter() - t_addsms
        daughter_combo_time += time.perf_counter() - t_prod

    phase['top_level_product'] = daughter_combo_time
    phase['copy_and_attach'] = attach_time
    phase['set_global_properties'] = global_time
    phase['addSMS'] = addsms_time

    if massCompress or invisibleCompress:
        t0 = time.perf_counter()
        smsTopDict.compress(massCompress, invisibleCompress, minmassgap, minmassgapISR)
        phase['compress'] = time.perf_counter() - t0

    phase['total'] = time.perf_counter() - t_total
    return smsTopDict, phase, counters

phase_model = _fresh_model()
timed_topDict, phase_times, phase_counts = timed_decomposeNew(
    phase_model,
    sigmacut,
    massCompress=massCompress,
    invisibleCompress=invisibleCompress,
    minmassgap=mingap,
    minmassgapISR=mingapISR,
 )

print(f"timed decomposeNew in {phase_times['total']:.3f} s")
print(f"topologies={len(timed_topDict)} sms={len(timed_topDict.getSMSList())}")
print("\nPhase timings")
print("-" * 54)
for name, value in phase_times.items():
    print(f"{name:<24} {value:10.3f} s")

print("\nCounters")
print("-" * 54)
for name, value in phase_counts.items():
    print(f"{name:<24} {value:10}")

inside_total = (
    phase_times['build_primary_sms']
    + phase_times['build_subtree_cache']
    + phase_times['copy_and_attach']
    + phase_times['set_global_properties']
    + phase_times['addSMS']
    + phase_times['compress']
 )
print("\nInside-loop split")
print("-" * 54)
print(f"top_level_product total      {phase_times['top_level_product']:10.3f} s")
print(f"  copy_and_attach            {phase_times['copy_and_attach']:10.3f} s")
print(f"  set_global_properties      {phase_times['set_global_properties']:10.3f} s")
print(f"  addSMS                     {phase_times['addSMS']:10.3f} s")
print(f"  residual/product overhead  {phase_times['top_level_product'] - phase_times['copy_and_attach'] - phase_times['set_global_properties'] - phase_times['addSMS']:10.3f} s")

INFO in model.updateParticles() in 428: Loaded 58 BSM particles


timed decomposeNew in 41.115 s
topologies=42 sms=28077

Phase timings
------------------------------------------------------
xsec_sort                     0.001 s
build_primary_sms             0.020 s
build_subtree_cache           0.352 s
top_level_product            18.164 s
copy_and_attach               2.471 s
set_global_properties         4.752 s
addSMS                        6.849 s
compress                     22.576 s
total                        41.115 s

Counters
------------------------------------------------------
production_sms                   38
cache_particles                   6
primary_combos_considered    6937513
primary_combos_kept           25541

Inside-loop split
------------------------------------------------------
top_level_product total          18.164 s
  copy_and_attach                 2.471 s
  set_global_properties           4.752 s
  addSMS                          6.849 s
  residual/product overhead       4.093 s


In [ ]:
import smodels.decomposition.topologyDict as topologyDict_mod
import smodels.decomposition.theorySMS as theorySMS_mod

def instrumented_decomposeNew_run():
    timings = {
        'addSMS_total': 0.0,
        'addSMS_compareTo': 0.0,
        'addSMS_merge_add': 0.0,
        'setGlobalProperties_total': 0.0,
        'setGlobal_computeCanonName': 0.0,
        'setGlobal_sort': 0.0,
        'setGlobal_weight': 0.0,
        'massCompress_total': 0.0,
        'invisibleCompress_total': 0.0,
    }
    counters = {
        'addSMS_calls': 0,
        'addSMS_newcanon': 0,
        'addSMS_merges': 0,
        'addSMS_inserts': 0,
        'compareTo_calls': 0,
        'setGlobal_calls': 0,
        'massCompress_calls': 0,
        'invisibleCompress_calls': 0,
    }

    original_addSMS = topologyDict_mod.TopologyDict.addSMS
    original_compareTo = theorySMS_mod.TheorySMS.compareTo
    original_add = theorySMS_mod.TheorySMS.__add__
    original_setGlobalProperties = theorySMS_mod.TheorySMS.setGlobalProperties
    original_massCompress = theorySMS_mod.TheorySMS.massCompress
    original_invisibleCompress = theorySMS_mod.TheorySMS.invisibleCompress

    def wrapped_compareTo(self, other):
        t0 = time.perf_counter()
        try:
            return original_compareTo(self, other)
        finally:
            timings['addSMS_compareTo'] += time.perf_counter() - t0
            counters['compareTo_calls'] += 1

    def wrapped_add(self, other):
        t0 = time.perf_counter()
        try:
            return original_add(self, other)
        finally:
            timings['addSMS_merge_add'] += time.perf_counter() - t0

    def wrapped_setGlobalProperties(self, sort=True, canonName=True, weight=True):
        t0_total = time.perf_counter()
        counters['setGlobal_calls'] += 1
        if canonName:
            t0 = time.perf_counter()
            self._canonName = self.computeCanonName()
            timings['setGlobal_computeCanonName'] += time.perf_counter() - t0
        if sort:
            t0 = time.perf_counter()
            self.sort(force=True)
            timings['setGlobal_sort'] += time.perf_counter() - t0
        if weight:
            t0 = time.perf_counter()
            self.weightList = self.computeWeightList()
            timings['setGlobal_weight'] += time.perf_counter() - t0
        timings['setGlobalProperties_total'] += time.perf_counter() - t0_total

    def wrapped_addSMS(self, newSMS):
        t0 = time.perf_counter()
        counters['addSMS_calls'] += 1
        try:
            if isinstance(newSMS, theorySMS_mod.TheorySMS):
                canonName = newSMS.canonName
                if canonName not in self:
                    counters['addSMS_newcanon'] += 1
                else:
                    smsList = self[canonName]
                    lo = 0
                    hi = len(smsList)
                    cmp = None
                    while lo < hi:
                        mid = (lo + hi) // 2
                        cmp = smsList[mid].compareTo(newSMS)
                        if cmp < 0:
                            lo = mid + 1
                        elif cmp > 0:
                            hi = mid
                        else:
                            lo = mid
                            break
                    if cmp == 0:
                        counters['addSMS_merges'] += 1
                    else:
                        counters['addSMS_inserts'] += 1
            return original_addSMS(self, newSMS)
        finally:
            timings['addSMS_total'] += time.perf_counter() - t0

    def wrapped_massCompress(self, minmassgap, minmassgapISR):
        t0 = time.perf_counter()
        counters['massCompress_calls'] += 1
        try:
            return original_massCompress(self, minmassgap, minmassgapISR)
        finally:
            timings['massCompress_total'] += time.perf_counter() - t0

    def wrapped_invisibleCompress(self):
        t0 = time.perf_counter()
        counters['invisibleCompress_calls'] += 1
        try:
            return original_invisibleCompress(self)
        finally:
            timings['invisibleCompress_total'] += time.perf_counter() - t0

    topologyDict_mod.TopologyDict.addSMS = wrapped_addSMS
    theorySMS_mod.TheorySMS.compareTo = wrapped_compareTo
    theorySMS_mod.TheorySMS.__add__ = wrapped_add
    theorySMS_mod.TheorySMS.setGlobalProperties = wrapped_setGlobalProperties
    theorySMS_mod.TheorySMS.massCompress = wrapped_massCompress
    theorySMS_mod.TheorySMS.invisibleCompress = wrapped_invisibleCompress

    try:
        model = _fresh_model()
        t0 = time.perf_counter()
        result = decomposeNew(
            model,
            sigmacut,
            massCompress=massCompress,
            invisibleCompress=invisibleCompress,
            minmassgap=mingap,
            minmassgapISR=mingapISR,
        )
        total = time.perf_counter() - t0
    finally:
        topologyDict_mod.TopologyDict.addSMS = original_addSMS
        theorySMS_mod.TheorySMS.compareTo = original_compareTo
        theorySMS_mod.TheorySMS.__add__ = original_add
        theorySMS_mod.TheorySMS.setGlobalProperties = original_setGlobalProperties
        theorySMS_mod.TheorySMS.massCompress = original_massCompress
        theorySMS_mod.TheorySMS.invisibleCompress = original_invisibleCompress

    return result, total, timings, counters

instrumented_topDict, instrumented_total, instrumented_timings, instrumented_counts = instrumented_decomposeNew_run()

print(f"instrumented decomposeNew in {instrumented_total:.3f} s")
print(f"topologies={len(instrumented_topDict)} sms={len(instrumented_topDict.getSMSList())}")

print("\nInstrumented timings")
print("-" * 60)
for key, value in instrumented_timings.items():
    print(f"{key:<28} {value:10.3f} s")

print("\nInstrumented counters")
print("-" * 60)
for key, value in instrumented_counts.items():
    print(f"{key:<28} {value:10}")

print("\nDerived splits")
print("-" * 60)
print(f"addSMS non-compare overhead      {instrumented_timings['addSMS_total'] - instrumented_timings['addSMS_compareTo'] - instrumented_timings['addSMS_merge_add']:10.3f} s")
print(f"setGlobal non-subcall overhead  {instrumented_timings['setGlobalProperties_total'] - instrumented_timings['setGlobal_computeCanonName'] - instrumented_timings['setGlobal_sort'] - instrumented_timings['setGlobal_weight']:10.3f} s")

INFO in model.updateParticles() in 428: Loaded 58 BSM particles


instrumented decomposeNew in 40.312 s
topologies=42 sms=28077

Instrumented timings
------------------------------------------------------------
addSMS_total                     14.301 s
addSMS_compareTo                 10.817 s
addSMS_merge_add                  2.842 s
setGlobalProperties_total        12.077 s
setGlobal_computeCanonName        4.494 s
setGlobal_sort                    7.044 s
setGlobal_weight                  0.452 s
massCompress_total                4.356 s
invisibleCompress_total           6.594 s

Instrumented counters
------------------------------------------------------------
addSMS_calls                      46011
addSMS_newcanon                      42
addSMS_merges                     17934
addSMS_inserts                    28035
compareTo_calls                  906481
setGlobal_calls                   71833
massCompress_calls                53881
invisibleCompress_calls           66620

Derived splits
---------------------------------------------------------